## 1 Introduction
Bài nghiên cứu này tập trung vào việc làm rõ tầm ảnh hưởng của các đặc tính âm thanh kỹ thuật số đối với mức độ thành công (thương mại) của một tác phẩm âm nhạc.

*   **Câu hỏi nghiên cứu:** Các chỉ số âm nhạc như độ nhảy (`danceability_%`) hay năng lượng (`energy_%`) có tác động thuận chiều đến tổng lượt stream không?
*   **Giả thuyết nghiên cứu:**
    *   $H_0$: Độ nhảy và năng lượng không có tác động hoặc tác động nghịch chiều đến lượt stream.
    *   $H_1$: Độ nhảy và năng lượng có tác động thuận chiều tích cực đến lượt stream.

## 2 Mô tả dữ liệu và tiền xử lý

### a. Mô tả dữ liệu
#### Nguồn dữ liệu

Dataset: Top Spotify Songs 2023 Dataset (Kaggle)

Tập dữ liệu bao gồm các thông tin về những bài hát thành công nhất trên nền tảng Spotify trong năm 2023. Để phục vụ cho câu hỏi nghiên cứu về mối quan hệ giữa đặc tính âm thanh và mức độ thành công thương mại, các biến số được lựa chọn và phân loại cấu trúc như sau:

| Tên biến | Kiểu dữ liệu | Vai trò trong mô hình | Ý nghĩa chức năng |
| :--- | :--- | :--- | :--- |
| `streams` | Số nguyên (`int64`) | **Biến phụ thuộc** (Target - $y$) | Tổng số lượt nghe tích lũy trên Spotify. Đây là thước đo chính cho sự thành công thương mại. |
| `log_streams` | Số thực (`float64`) | **Biến phụ thuộc** (Sau biến đổi) | Giá trị Logarit tự nhiên của lượt stream ($\ln(1 + \text{streams})$), dùng để xử lý phân phối lệch và đảm bảo giả định hồi quy. |
| `danceability_%` | Số nguyên (`int64`) | **Biến độc lập** (Predictor - $x_1$) | Độ phù hợp của bài hát đối với việc nhảy nhảy (thang điểm 0 - 100%), dựa trên nhịp điệu, độ ổn định của nhịp. |
| `energy_%` | Số nguyên (`int64`) | **Biến độc lập** (Predictor - $x_2$) | Phép đo tốc độ và cường độ âm thanh (thang điểm 0 - 100%), đại diện cho độ sôi động, mạnh mẽ của bài hát. |
| `artist(s)_name` | Chuỗi ký tự (`str`) | Biến phân loại / Định danh | Tên (các) nghệ sĩ trình bày bài hát. |
| `released_year` | Số nguyên (`int64`) | Biến thời gian / Kiểm soát | Năm bài hát chính thức được phát hành. |

*Lưu ý về thang đo:* Hai biến độc lập chính (`danceability_%` và `energy_%`) được Spotify chuẩn hóa dưới dạng tỷ lệ phần trăm (%), giúp mô hình hồi quy OLS dễ giải thích ý nghĩa hệ số biên tế ($\beta$) hơn.

#### b. Tiền xử lý: Import thư viện và làm sạch dữ liệu

In [1]:
# Auto-generated code for data analysis after cleaning
%run data_cleaning.ipynb

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Read the cleaned data
df = pd.read_csv("../datasets/processed/cleaned_spotify_2023.csv")
print(f"Dataset size for analysis: {df.shape}")

--- Downloading origin dataset ---
Initial dataset size: 953 rows, 24 columns

<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   track_name            953 non-null    str  
 1   artist(s)_name        953 non-null    str  
 2   artist_count          953 non-null    int64
 3   released_year         953 non-null    int64
 4   released_month        953 non-null    int64
 5   released_day          953 non-null    int64
 6   in_spotify_playlists  953 non-null    int64
 7   in_spotify_charts     953 non-null    int64
 8   streams               953 non-null    str  
 9   in_apple_playlists    953 non-null    int64
 10  in_apple_charts       953 non-null    int64
 11  in_deezer_playlists   953 non-null    str  
 12  in_deezer_charts      953 non-null    int64
 13  in_shazam_charts      903 non-null    str  
 14  bpm                   953 non-null    

In [3]:
selected_columns = ['streams', 'log_streams', 'danceability_%', 'energy_%', 'released_year']

print("--- Statistical Summary of Key Variables ---")
# Check data types and count of non-null values
print(df[selected_columns].info())

# Display mathematical distribution (Min, Max, Mean) for EDA preparation
display(df[selected_columns].describe().round(2))

--- Statistical Summary of Key Variables ---
<class 'pandas.DataFrame'>
RangeIndex: 952 entries, 0 to 951
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   streams         952 non-null    int64  
 1   log_streams     952 non-null    float64
 2   danceability_%  952 non-null    int64  
 3   energy_%        952 non-null    int64  
 4   released_year   952 non-null    int64  
dtypes: float64(1), int64(4)
memory usage: 37.3 KB
None


,streams,log_streams,danceability_%,energy_%,released_year
count,9.520000e+02,952.00,952.00,952.00,952.00
mean,5.141374e+08,19.51,66.98,64.27,2018.29
std,5.668569e+08,1.15,14.63,16.56,11.01
min,2.762000e+03,7.92,23.00,9.00,1930.00
25%,1.416362e+08,18.77,57.00,53.00,2020.00
50%,2.905309e+08,19.49,69.00,66.00,2022.00
75%,6.738690e+08,20.33,78.00,77.00,2022.00
max,3.703895e+09,22.03,96.00,97.00,2023.00


## 3 Phân tích khám phá dữ liệu (EDA)
* Phân tích đơn biến
    * Histogram phân phối streams
    * Boxplot phát hiện outlier
    * Thống kê mô tả các biến số
* Phân tích đa biến
    * Scatter plot:
    * danceability_% vs streams
    * energy_% vs streams
    * Correlation heatmap

## 4 Kiểm định giả thuyết & Mô hình hóa
* Tương quan
* Pearson correlation
* Spearman correlation
* Hồi quy tuyến tính

        y=β0+β1x1+β2x2+ϵ

* Trong đó:

    * y: số lượt stream
    * x1: danceability
    * x2: energy

* Đánh giá:
    * hệ số hồi quy
    * p-value
    * R² score

## 5 Kết luận & Giải thích kết quả (Conclusion & Discussion)
* Trả lời trực tiếp câu hỏi nghiên cứu: Giả thuyết thuận chiều là Đúng hay Sai dựa trên dữ liệu thực tế.
* Góc nhìn thực tế: Tại sao lại có kết quả đó? (Ví dụ: Nếu không thuận chiều, có thể vì năm 2023 người nghe chuộng nhạc lofi, chillout hơn là nhạc nhảy sôi động).
* Hạn chế của đề tài: Dữ liệu chỉ gói gọn trong năm 2023, chưa tính đến yếu tố marketing, độ nổi tiếng sẵn có của nghệ sĩ (như Taylor Swift, The Weeknd).


## 6 Ý nghĩa thực tiễn (Insights)

Kết quả nghiên cứu có thể giúp:

* Nghệ sĩ tối ưu phong cách âm nhạc
* Producer hiểu xu hướng thị hiếu
* Xây dựng mô hình dự đoán độ phổ biến bài hát
* Hỗ trợ recommendation system cho nền tảng streaming